# SageMaker Training Job

Tento notebook ukazuje, ako spustiť SageMaker Training Job pre house price prediction model.

## Prerekvizity

1. ✅ Deployed SageMaker infrastructure (Terraform)
2. ✅ Training data uploaded to S3
3. ✅ Docker image built and pushed to ECR
4. ✅ MLflow tracking server running (optional)

## Postup

1. Setup AWS credentials a region
2. Upload training data do S3
3. Build a push Docker image do ECR
4. Launch SageMaker Training Job
5. Monitor job progress
6. Check results v MLflow

## 1. Setup

In [ ]:
import os
import json
import boto3
import sagemaker
from sagemaker.estimator import Estimator
from datetime import datetime
import subprocess

# AWS Session
boto_session = boto3.Session(region_name='eu-west-1')
sagemaker_session = sagemaker.Session(boto_session=boto_session)

# Region
region = boto_session.region_name
print(f"AWS Region: {region}")

## 2. Load Terraform Outputs

Načítame ARN role, S3 buckety a ECR repository URL z Terraform outputs.

In [ ]:
def get_terraform_output(key: str, tf_dir: str = "../infra/terraform/sagemaker") -> str:
    """Získaj hodnotu z Terraform output."""
    result = subprocess.run(
        ["terraform", "output", "-raw", key],
        cwd=tf_dir,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to get Terraform output '{key}': {result.stderr}")
    return result.stdout.strip()

# Get Terraform outputs
role_arn = get_terraform_output("sagemaker_execution_role_arn")
s3_data_bucket = get_terraform_output("s3_data_bucket_name")
s3_model_bucket = get_terraform_output("s3_model_bucket_name")
ecr_repository_url = get_terraform_output("ecr_repository_url")

print(f"SageMaker Role ARN: {role_arn}")
print(f"S3 Data Bucket: {s3_data_bucket}")
print(f"S3 Model Bucket: {s3_model_bucket}")
print(f"ECR Repository: {ecr_repository_url}")

## 3. Upload Training Data to S3

Upload train.csv do S3 data bucketu.

In [ ]:
# Local data path
local_train_path = "../data/train.csv"

# S3 paths
s3_train_prefix = "data"
s3_train_path = f"s3://{s3_data_bucket}/{s3_train_prefix}/train.csv"

# Upload to S3
print(f"Uploading {local_train_path} to {s3_train_path}...")
s3_client = boto3.client('s3', region_name=region)
s3_client.upload_file(local_train_path, s3_data_bucket, f"{s3_train_prefix}/train.csv")
print("✓ Data uploaded successfully!")

## 4. Build and Push Docker Image to ECR

Build Docker image a push do ECR repository.

**Note:** Tento krok môžeš spustiť aj z terminálu:
```bash
cd ..
./scripts/build_and_push.sh eu-west-1
```

In [ ]:
# Build and push using the script
build_script = "../scripts/build_and_push.sh"

print("Building and pushing Docker image to ECR...")
print("This may take 5-10 minutes...")

result = subprocess.run(
    ["bash", build_script, region],
    cwd="..",
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Docker image built and pushed successfully!")
    print(result.stdout)
else:
    print("✗ Failed to build/push Docker image:")
    print(result.stderr)
    raise RuntimeError("Docker build failed")

## 5. Configure Training Job

Nastav hyperparametre a konfiguráciu pre SageMaker Training Job.

In [ ]:
# Training job name (musí byť unique)
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
training_job_name = f"house-price-training-{timestamp}"

# Docker image (latest tag)
training_image = f"{ecr_repository_url}:latest"

# S3 output path pre model artifacts
s3_output_path = f"s3://{s3_model_bucket}/training-jobs/{training_job_name}"

# Hyperparametre
hyperparameters = {
    "n-estimators": 500,
    "max-depth": 6,
    "learning-rate": 0.05,
    "subsample": 0.8,
    "colsample-bytree": 0.8,
    "min-child-weight": 3,
    "reg-alpha": 0.1,
    "reg-lambda": 1.0,
}

# MLflow tracking (optional - nastav ak máš MLflow server)
# hyperparameters["mlflow-tracking-uri"] = "http://your-mlflow-server.com"
# hyperparameters["mlflow-experiment-name"] = "house-prices-sagemaker"

print(f"Training Job Name: {training_job_name}")
print(f"Training Image: {training_image}")
print(f"S3 Output: {s3_output_path}")
print(f"Hyperparameters: {json.dumps(hyperparameters, indent=2)}")

## 6. Create SageMaker Estimator

In [ ]:
# Create Estimator
estimator = Estimator(
    image_uri=training_image,
    role=role_arn,
    instance_count=1,
    instance_type="ml.m5.xlarge",  # 4 vCPUs, 16 GB RAM, ~$0.23/hour
    output_path=s3_output_path,
    sagemaker_session=sagemaker_session,
    hyperparameters=hyperparameters,
    base_job_name="house-price-training",
    max_run=3600,  # Max 1 hour
    use_spot_instances=True,  # Použij Spot instances pre úsporu (~70% discount)
    max_wait=7200,  # Max wait for spot (2 hours)
)

print("✓ Estimator created successfully!")

## 7. Launch Training Job

Spustí SageMaker Training Job.

**Note:** Training trvá približne **5-10 minút** (v závislosti od instance type a dát).

In [ ]:
# Input data channels
inputs = {
    "train": f"s3://{s3_data_bucket}/{s3_train_prefix}"
}

print(f"Starting training job: {training_job_name}")
print(f"Input data: {inputs['train']}")
print("=" * 80)
print("Training job launched! Logs will appear below...")
print("=" * 80)

# Spustí training job (wait=True znamená, že čaká na dokončenie)
estimator.fit(inputs=inputs, wait=True, logs="All")

print("=" * 80)
print("✓ Training job completed successfully!")
print("=" * 80)

## 8. Check Training Job Details

In [ ]:
# Get training job details
sagemaker_client = boto3.client('sagemaker', region_name=region)
job_details = sagemaker_client.describe_training_job(TrainingJobName=estimator.latest_training_job.name)

print(f"Training Job Name: {job_details['TrainingJobName']}")
print(f"Status: {job_details['TrainingJobStatus']}")
print(f"Training Time: {job_details['TrainingTimeInSeconds']}s")
print(f"Billable Time: {job_details['BillableTimeInSeconds']}s")
print(f"Instance Type: {job_details['ResourceConfig']['InstanceType']}")
print(f"Model Artifacts: {job_details['ModelArtifacts']['S3ModelArtifacts']}")

# Final metrics (ak sú dostupné)
if 'FinalMetricDataList' in job_details:
    print("\nFinal Metrics:")
    for metric in job_details['FinalMetricDataList']:
        print(f"  {metric['MetricName']}: {metric['Value']:.6f}")

## 9. Download Model Artifacts (Optional)

In [ ]:
# Download model.tar.gz from S3
model_s3_uri = job_details['ModelArtifacts']['S3ModelArtifacts']
local_model_path = f"../models/{training_job_name}/model.tar.gz"

print(f"Downloading model from {model_s3_uri}...")

# Parse S3 URI
s3_parts = model_s3_uri.replace("s3://", "").split("/", 1)
bucket = s3_parts[0]
key = s3_parts[1]

# Create local directory
os.makedirs(os.path.dirname(local_model_path), exist_ok=True)

# Download
s3_client.download_file(bucket, key, local_model_path)
print(f"✓ Model downloaded to: {local_model_path}")

# Extract model.tar.gz
import tarfile
extract_dir = os.path.dirname(local_model_path)
with tarfile.open(local_model_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)
print(f"✓ Model extracted to: {extract_dir}")
print(f"  Contents: {os.listdir(extract_dir)}")

## 10. Check MLflow (Optional)

Ak si nastavil MLflow tracking URI, môžeš skontrolovať experiment v MLflow UI.

In [ ]:
# Nastav MLflow tracking URI (ak máš MLflow server)
# import mlflow
# mlflow.set_tracking_uri("http://your-mlflow-server.com")

# # Search for runs
# experiment = mlflow.get_experiment_by_name("house-prices-sagemaker")
# if experiment:
#     runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
#     print(f"Found {len(runs)} runs in experiment '{experiment.name}'")
#     print(runs[["run_id", "metrics.rmse", "metrics.r2"]].head())
# else:
#     print("Experiment not found in MLflow")

print("To view runs in MLflow UI, visit: http://your-mlflow-server.com")

## 11. Next Steps

Po úspešnom tréningu môžeš:

1. **Deploy model to SageMaker Endpoint** (real-time inference)
   ```python
   predictor = estimator.deploy(
       initial_instance_count=1,
       instance_type="ml.t2.medium"
   )
   ```

2. **Register model to MLflow Model Registry**
   - Transition to Staging/Production

3. **Run Batch Transform** (batch predictions)
   ```python
   transformer = estimator.transformer(
       instance_count=1,
       instance_type="ml.m5.xlarge"
   )
   transformer.transform(data=s3_test_data, content_type="text/csv")
   ```

4. **Hyperparameter Tuning** (HPO)
   ```python
   from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter
   
   hyperparameter_ranges = {
       "n-estimators": IntegerParameter(100, 1000),
       "max-depth": IntegerParameter(3, 10),
       "learning-rate": ContinuousParameter(0.01, 0.3),
   }
   
   tuner = HyperparameterTuner(
       estimator,
       objective_metric_name="validation:rmse",
       hyperparameter_ranges=hyperparameter_ranges,
       max_jobs=20,
       max_parallel_jobs=2
   )
   tuner.fit(inputs)
   ```

5. **Setup CI/CD Pipeline**
   - GitHub Actions workflow pre automatický training
   - Automatic deployment on new data

See notebooks 05-08 for more details!

## Cost Estimate

**Training Job (ml.m5.xlarge):**
- Instance: $0.23/hour
- Training time: ~10 minutes
- Cost per job: ~$0.04
- With Spot Instances (~70% discount): ~$0.01

**S3 Storage:**
- Data: ~1 MB (train.csv)
- Model artifacts: ~10 MB per job
- Cost: ~$0.001/month

**ECR Storage:**
- Docker image: ~500 MB
- Cost: ~$0.05/month

**Total for 10 training jobs:** ~$0.10-0.40 + $0.05/month storage